# Le long chemin pour avoir les génomes RefSeq ET leur taxonomie

In [2]:
import yaml
from Bio import SeqIO, Entrez
import os
import pandas as pd
from io import StringIO

1. télécharger la table d'assemblage
2. filtrage par 'Complet genome' pour la colonne assembly_level
3. récupération adresse ftp
4. téléchargement et dézippage
5. ouverture fichier, et récupération de l'accession (code exemple NZ_LT667500)
6. requete vers NCBI (National Center for Biotechnology Information) pour avoir la taxonomie en fonction de l'accession

PB du passage à l'échelle : si trop de requete vers NCBI ... parallélisation du traitement (wget ...), api_key pour augmenter le nb requete /seconde

In [3]:
output_dir = './temp_refseq'
os.makedirs(output_dir, exist_ok=True)

!wget  -P {output_dir} https://ftp.ncbi.nlm.nih.gov/genomes/refseq/bacteria/assembly_summary.txt



7[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[https://ftp.ncbi.nlm.nih.gov/g]87Saving './temp_refseq/assembly_summary.txt'
87assembly_summary.txt   0% [<=>                           ]   31.63K    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [ <=>                          ]    1.21M    1.17MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [  <=>                         ]    2.50M    1.23MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [   <=>                        ]    3.92M    1.29MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [    <=>                       ]    5.41M    1.34MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [     <=>                      ]    6.64M    1.32MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembly_summary.txt   0% [      <=>                     ]    7.62M    1.26MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87assembl

In [4]:
# conversion en csv / Dataframe
with open(f"{output_dir}/assembly_summary.txt", "r") as file:
    lines = file.readlines()[1:]  # Supprime la première ligne
    lines[0] = lines[0].lstrip("#")  # Enlève le '#' du début de l'en-tête

df = pd.read_csv(StringIO("".join(lines)), sep="\t", low_memory=False) # Convertir en DataFrame directement
df.to_csv(f"{output_dir}/assembly_summary.csv", sep="\t", index=False) # Sauvegarder en CSV propre

### on ne garde que les genomes complets.  assembly_level == "Complete Genome"

In [13]:

pd.set_option('display.width', 300)  # Augmente la largeur totale de l'affichage
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes complètement

csv_file_in = f"{output_dir}/assembly_summary.csv"
csv_file_out = f"{output_dir}/assembly_summary_filtered_Complete_Genome.csv"


info_all_genome_df = pd.read_csv(csv_file_in, sep="\t",low_memory=False, usecols=["assembly_accession", "assembly_level",'refseq_category', "ftp_path"])
df_completed = info_all_genome_df[
    (info_all_genome_df["assembly_level"] == "Complete Genome") &  # Filtre "Complete Genome"
    (info_all_genome_df["ftp_path"].notna()) &                     # Supprime les "na"
    (info_all_genome_df["ftp_path"].str.startswith("https"))       # Garde uniquement les URL HTTPS
].drop(columns=["assembly_level"]).reset_index(drop=True)
df_completed.to_csv(csv_file_out, sep="\t", index=False)

print(f"Filter all (nb {len(info_all_genome_df) })by 'Complete genome' in assembly level (nb {len(df_completed)})")
print(df_completed)


Filter all (nb 410259)by 'Complete genome' in assembly level (nb 47367)
      assembly_accession   refseq_category                                                                                    ftp_path
0        GCF_900128725.1                na  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0
1        GCF_003044255.1                na        https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/003/044/255/GCF_003044255.1_ASM304425v1
2        GCF_009730575.1                na        https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/009/730/575/GCF_009730575.1_ASM973057v1
3        GCF_016406305.1                na       https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/016/406/305/GCF_016406305.1_ASM1640630v1
4        GCF_016406325.1  reference genome       https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/016/406/325/GCF_016406325.1_ASM1640632v1
...                  ...               ...                                                                            

### un génome référent par espèce, refseq_category== "reference genome", voir https://www.ncbi.nlm.nih.gov/refseq/about/prokaryotes/

In [15]:
df_referenced = df_completed[(df_completed["refseq_category"] == "reference genome") ].drop(columns=["refseq_category"]).reset_index(drop=True)
df_referenced 

,assembly_accession,ftp_path
0,GCF_016406325.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/016/406/325/GCF_016406325.1_ASM1640632v1
1,GCF_001027285.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/027/285/GCF_001027285.1_ASM102728v1
2,GCF_001189295.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/189/295/GCF_001189295.1_ASM118929v1
3,GCF_022870945.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/022/870/945/GCF_022870945.1_ASM2287094v1
4,GCF_002222655.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/002/222/655/GCF_002222655.1_ASM222265v1
...,...,...
5766,GCF_019720755.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/019/720/755/GCF_019720755.1_ASM1972075v1
5767,GCF_041154365.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/041/154/365/GCF_041154365.1_ASM4115436v1
5768,GCF_030866785.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/030/866/785/GCF_030866785.1_ASM3086678v1
5769,GCF_031656915.1,https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/031/656/915/GCF_031656915.1_ASM3165691v1


In [24]:
assembly_accession,refseq_category, https_path = df_completed.loc[0]
print(f"assembly_accession: {assembly_accession}, refseq_category: {refseq_category}, https_path {https_path}")
ftp_path = https_path[8:]
end_url_file = ftp_path.split('/')[-1]
real_ftp_path = f"{ftp_path}/{end_url_file}_genomic.fna.gz"
file_path = f"{output_dir}/{end_url_file}_genomic.fna.gz"
print("ftp url :", real_ftp_path)
os.system(f"wget -P {output_dir} {ftp_path}/{end_url_file}_genomic.fna.gz")
print("download filename ", file_path)
os.system(f"gzip -d {file_path}")
file_path = file_path.removesuffix(".gz")
print("unzipped filename ", file_path)


assembly_accession: GCF_900128725.1, refseq_category: na, https_path https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0
ftp url : ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz
HSTS in effect for ftp.ncbi.nlm.nih.gov:80
[0] Downloading 'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz' ...
Saving './temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz.2'
HTTP response 200 OK [https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/900/128/725/GCF_900128725.1_BCifornacula_v1.0/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz]
download filename  ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna.gz
unzipped filename  ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna


gzip: ./temp_refseq/GCF_900128725.1_BCifornacula_v1.0_genomic.fna already exists;	not overwritten


In [18]:
with open(file_path, "r", encoding='utf-8') as reader:
    first_line = reader.readline()
    print(first_line)
    accession = first_line.split('.')[0][1:]

>NZ_LT667500.1 Buchnera aphidicola strain BCifornacula voucher 2912 chromosome 1



In [19]:
yaml_file = "../import_dataset/refseq/NCBI_credentials.yaml"

with open(yaml_file, 'r') as file:
    credentials = yaml.safe_load(file)

Entrez.email = credentials.get("email")  # Remplacez par votre email
Entrez.api_key = credentials.get("api_key") #""
Entrez.max_tries = 5
Entrez.sleep_between_tries = 15

In [20]:
with Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text") as taxo_handle:
    x = SeqIO.read(taxo_handle, 'genbank')
    classif = x.annotations['taxonomy']
    sub = x.annotations['organism']

print(classif)

['Bacteria', 'Pseudomonadati', 'Pseudomonadota', 'Gammaproteobacteria', 'Enterobacterales', 'Erwiniaceae', 'Buchnera']


In [21]:
TAXO_LEVELS = ["domain", "phylum", "group", "order", "family", "specie"] 
# From Siegfried
order: int | str = 0
for e in classif:
    if e[-4:] == 'ales':
        order = e
if order:
    group = classif[2] if classif[2][-4:] != 'ales' else classif[1]
    taxo_list = [classif[0], classif[1],  group, order, sub.split(' ')[0], sub.split(' ')[1]]
    print(*list(zip(TAXO_LEVELS, taxo_list)), sep='\n')
    file_name: str = f"{output_dir}/{"_".join(taxo_list)}.fna"
    print(f' change filename to  {file_name}")')
else:
    print(f' rm  {file_path}")')

('domain', 'Bacteria')
('phylum', 'Pseudomonadati')
('group', 'Pseudomonadota')
('order', 'Enterobacterales')
('family', 'Buchnera')
('specie', 'aphidicola')
 change filename to  ./temp_refseq/Bacteria_Pseudomonadati_Pseudomonadota_Enterobacterales_Buchnera_aphidicola.fna")


In [ ]:
# GET TAXO BY BATCH
# accession_map = {}

# with open(decompressed_path, "r", encoding='utf-8') as reader:
#             # first_line = reader.readline().strip()
#             first_line = reader.readline()
#             accession = first_line.split('.')[0][1:]  # Extraction de l'accession
#             accession_map[accession] = decompressed_path
#             decompressed_files.append(decompressed_path)

# accessions = list(accession_map.keys())
# for i in tqdm(range(0, len(accessions), batch_size), desc="Fetching taxonomy data"):
#     batch = accessions[i:i + batch_size]
#     try:1
#         with Entrez.efetch(db="nucleotide", id=batch, rettype="gb", retmode="text") as taxo_handle:
#             # records = SeqIO.read(taxo_handle, 'genbank')
#             records = SeqI%ùO.par